![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 02: Python)**

**LabClass M02: Text Files and Reusable Python Functions**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find an issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

## LabClass M02: Text Files and Reusable Python Functions

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This lab class is the common foundation for the A, B, and C LabClasses streams. It practises file input, output, filtering, and reusable functions before later data-processing labs.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>A filtered baby-name summary file and reusable extraction function</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development. Not directly assessed.</td>
</tr>
</tbody>
</table>

</div>

---


**Table of Contents**

- [1. Overview and Learning Goals](#m02-overview)
- [2. Setup and Background](#m02-setup)
- [3. Core Concepts](#m02-core-concepts)
- [4. Guided Implementation](#m02-guided-implementation)
- [5. Testing and Analysis](#m02-testing)
- [6. Student Tasks](#m02-student-tasks)
- [7. Reflection and References](#m02-reflection)


<a id="m02-overview"></a>

### 1. Overview and Learning Goals

This lab class uses USA baby-name text files to practise reading multiple files, extracting selected rows, writing a summary file, and wrapping the workflow in a reusable Python function.

By the end of this lab, students should be able to:

1. load multiple text files from public unit data or a local repository clone;
2. parse comma-separated text rows safely with `csv.DictReader`;
3. filter records by gender and name prefix;
4. write a clean summary file for later processing;
5. package the workflow as a reusable Python function.


<a id="m02-setup"></a>

### 2. Setup and Background

Use **Option A** in Google Colab or another online notebook environment. Use **Option B** only when you have cloned the SIT742 repository locally.

The setup cell below first looks for a local `Jupyter/data/` folder. If it cannot find one, it downloads the required public files from GitHub into the notebook runtime.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import csv
import fileinput
import re

PUBLIC_DATA_BASE = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
BABY_NAME_FILES = [
    "baby_name_2000.txt",
    "baby_name_2002.txt",
    "baby_name_2004.txt",
    "baby_name_2006.txt",
    "baby_name_2008.txt",
]

DOWNLOAD_DIR = Path("data") / "m02_baby_names"
LOCAL_DATA_DIRS = [
    Path("../data"),
    Path("Jupyter/data"),
    Path("SIT742/Jupyter/data"),
]

def resolve_or_download(filename):
    for data_dir in LOCAL_DATA_DIRS:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate

    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    target = DOWNLOAD_DIR / filename
    if not target.exists():
        urlretrieve(f"{PUBLIC_DATA_BASE}/{filename}", target)
    return target

baby_name_paths = [resolve_or_download(filename) for filename in BABY_NAME_FILES]
print("Files ready:")
for path in baby_name_paths:
    print("-", path)


<a id="m02-core-concepts"></a>

### 3. Core Concepts

The baby-name files use a simple CSV-style text format:

| year | name | gender | count |
| --- | --- | --- | --- |
| 2000 | Emily | F | 25957 |

The workflow is:

1. resolve the input files;
2. parse rows as dictionaries;
3. filter records;
4. sort the result;
5. write a summary file.


<a id="m02-guided-implementation"></a>

### 4. Guided Implementation

Start by loading all rows into a list of dictionaries. A dictionary representation makes the code easier to read than positional indexes such as `row[1]`.


In [ ]:
def load_baby_name_rows(paths):
    rows = []
    for path in paths:
        with path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                rows.append({
                    "year": int(row["year"]),
                    "name": row["name"],
                    "gender": row["gender"],
                    "count": int(row["count"]),
                    "source_file": path.name,
                })
    return rows

baby_rows = load_baby_name_rows(baby_name_paths)
print("Total rows:", len(baby_rows))
baby_rows[:5]


In [ ]:
OUTPUT_DIR = Path("outputs") / "m02_baby_names"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def write_name_summary(rows, output_file, gender=None, start_letter=None, top_n=None):
    filtered = []
    for row in rows:
        if gender and row["gender"] != gender:
            continue
        if start_letter and not row["name"].upper().startswith(start_letter.upper()):
            continue
        filtered.append(row)

    filtered = sorted(filtered, key=lambda row: row["count"], reverse=True)
    if top_n is not None:
        filtered = filtered[:top_n]

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with output_file.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["year", "name", "gender", "count"])
        writer.writeheader()
        writer.writerows({key: row[key] for key in ["year", "name", "gender", "count"]} for row in filtered)
    return filtered, output_file

top_a_names, summary_path = write_name_summary(
    baby_rows,
    OUTPUT_DIR / "female_a_names_top_20.csv",
    gender="F",
    start_letter="A",
    top_n=20,
)

print("Wrote:", summary_path)
top_a_names[:10]


The function below combines file resolution, parsing, filtering, sorting, and writing. It is the reusable version of the workflow.


In [ ]:
def extract_names(input_files, output_file, gender=None, start_letter=None, top_n=None):
    input_paths = [Path(path) for path in input_files]
    rows = load_baby_name_rows(input_paths)
    filtered, written_path = write_name_summary(
        rows,
        output_file,
        gender=gender,
        start_letter=start_letter,
        top_n=top_n,
    )
    return {
        "rows_read": len(rows),
        "rows_written": len(filtered),
        "output_file": written_path,
        "preview": filtered[:5],
    }

result = extract_names(
    baby_name_paths,
    OUTPUT_DIR / "male_b_names_top_15.csv",
    gender="M",
    start_letter="B",
    top_n=15,
)
result


<a id="m02-testing"></a>

### 5. Testing and Analysis

Use small checks to confirm that the workflow read the expected files, wrote an output file, and kept the filtering conditions.


In [ ]:
assert len(baby_name_paths) == 5
assert len(baby_rows) > 0
assert summary_path.exists()
assert all(row["gender"] == "F" for row in top_a_names)
assert all(row["name"].startswith("A") for row in top_a_names)
assert top_a_names == sorted(top_a_names, key=lambda row: row["count"], reverse=True)

print("M02 checks passed.")


<a id="m02-student-tasks"></a>

### 6. Student Tasks

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td>Write a file containing the top 25 female names starting with `E`.</td>
<td>Practises filtering and sorted output.</td>
<td>A CSV file path and a five-row preview.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td>Modify the workflow so it can filter a year range.</td>
<td>Extends a function without duplicating parsing code.</td>
<td>A function call and a check that all years are in range.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td>Compare the most frequent names across two years.</td>
<td>Connects file processing to simple exploratory analysis.</td>
<td>A short table or written comparison.</td>
</tr>
</tbody>
</table>

</div>


In [ ]:
# TODO: Complete Task 1 to Task 3 here.
# Keep your output files under OUTPUT_DIR so the notebook remains rerunnable.


<a id="m02-reflection"></a>

### 7. Reflection and References

Reflection questions:

1. Why is it safer to parse the text files with `csv.DictReader` than to split strings manually?
2. What assumptions does the filtering function make about the input files?
3. How would the workflow change if the input files were very large?

#### Further Readings

- Python `csv` module: https://docs.python.org/3/library/csv.html
- Python `pathlib` module: https://docs.python.org/3/library/pathlib.html
- USA Social Security baby names data: https://www.ssa.gov/oact/babynames/
